# DSB Monitoring Summary CSV Aggregation and QC

This notebook is a stepwise version of `MD_PD_Analysis.Rmd`, intended to run **R** code in Jupyter (typically via **IRkernel**).

If you open this notebook and VS Code shows a Python kernel, switch the notebook kernel to **R (IRkernel)**.

## 0. Packages

This section loads the packages used for import, manipulation, and plotting.

- `tidyverse` provides `dplyr`, `readr`, `purrr`, `stringr`, and `ggplot2`.

In [ ]:
library(tidyverse)

# Make plots larger and labels more legible throughout the report
knitr::opts_chunk$set(
  fig.width = 16,
  fig.height = 8,
  dpi = 200,
  out.width = "100%",
  fig.align = "center"
)

# Ensure rmarkdown can find Pandoc.
# On Windows, Quarto bundles pandoc in: <quarto>/bin/tools/pandoc.exe.
if (!nzchar(Sys.which("pandoc"))) {
  quarto_exe <- Sys.which("quarto")
  if (nzchar(quarto_exe)) {
    quarto_tools <- normalizePath(
      file.path(dirname(quarto_exe), "tools"),
      winslash = "/",
      mustWork = FALSE
    )
    if (dir.exists(quarto_tools)) {
      Sys.setenv(RSTUDIO_PANDOC = quarto_tools)
    }
  }
}

# Some users open the exported CSVs in Excel, which can lock the file on Windows.
# Use a safe writer that falls back to a timestamped filename if needed.
safe_write_csv <- function(df, path) {
  tryCatch(
    {
      readr::write_csv(df, path)
      path
    },
    error = function(e) {
      ts <- format(Sys.time(), "%Y%m%d_%H%M%S")
      alt <- sub("\\.csv$", paste0("_", ts, ".csv"), path)
      message(
        "Could not write '",
        path,
        "' (",
        conditionMessage(e),
        "). Writing to '",
        alt,
        "' instead."
      )
      readr::write_csv(df, alt)
      alt
    }
  )
}

# Toggle legacy normalization sections (currently unused in the streamlined report below).
RUN_LEGACY_NORMALIZATION <- FALSE

# Canonical 4-combo cis/trans definitions used throughout this report
combos_4 <- c("A_to_B", "C_to_D", "A_to_D", "C_to_B")
cis_combos <- c("A_to_B", "C_to_D")
trans_combos <- c("A_to_D", "C_to_B")

# PD controls used for normalization
SPIKEIN_ALLELE <- "CHRXV_L18_1PERCENT_CONTROL"
NOCUT_ALLELE <- "NO_CUT"
CONTROL_ALLELES <- c(SPIKEIN_ALLELE, NOCUT_ALLELE)

# Refactored summarizer: same schema for raw OR corrected counts.
# - Works on any data frame that has a `combo` column and a numeric count column.
# - Summarises over the canonical 4 combos, and computes cis/trans from combos.
summarize_combo4_counts_pct <- function(dat,
                                       count_col = "count",
                                       group_cols = c("batch", "time_point", "DSB"),
                                       combo_col = "combo") {
  if (!is.character(count_col) || length(count_col) != 1) {
    stop("count_col must be a single column name string")
  }
  if (!is.character(combo_col) || length(combo_col) != 1) {
    stop("combo_col must be a single column name string")
  }
  if (!all(group_cols %in% names(dat))) {
    missing <- setdiff(group_cols, names(dat))
    stop("Missing grouping columns in data: ", paste(missing, collapse = ", "))
  }
  if (!(combo_col %in% names(dat))) {
    stop("Missing combo column in data: ", combo_col)
  }
  if (!(count_col %in% names(dat))) {
    stop("Missing count column in data: ", count_col)
  }

  dat %>%
    filter(.data[[combo_col]] %in% combos_4) %>%
    group_by(across(all_of(group_cols))) %>%
    summarise(
      Total_Counts = sum(.data[[count_col]], na.rm = TRUE),
      Cis_Counts = sum(.data[[count_col]][.data[[combo_col]] %in% cis_combos], na.rm = TRUE),
      Trans_Counts = sum(.data[[count_col]][.data[[combo_col]] %in% trans_combos], na.rm = TRUE),
      A_to_B_Counts = sum(.data[[count_col]][.data[[combo_col]] == "A_to_B"], na.rm = TRUE),
      C_to_D_Counts = sum(.data[[count_col]][.data[[combo_col]] == "C_to_D"], na.rm = TRUE),
      A_to_D_Counts = sum(.data[[count_col]][.data[[combo_col]] == "A_to_D"], na.rm = TRUE),
      C_to_B_Counts = sum(.data[[count_col]][.data[[combo_col]] == "C_to_B"], na.rm = TRUE),
      Percent_Cis = if_else(Total_Counts > 0, 100 * Cis_Counts / Total_Counts, NA_real_),
      Percent_Trans = if_else(Total_Counts > 0, 100 * Trans_Counts / Total_Counts, NA_real_),
      Percent_A_to_B = if_else(Total_Counts > 0, 100 * A_to_B_Counts / Total_Counts, NA_real_),
      Percent_C_to_D = if_else(Total_Counts > 0, 100 * C_to_D_Counts / Total_Counts, NA_real_),
      Percent_A_to_D = if_else(Total_Counts > 0, 100 * A_to_D_Counts / Total_Counts, NA_real_),
      Percent_C_to_B = if_else(Total_Counts > 0, 100 * C_to_B_Counts / Total_Counts, NA_real_),
      .groups = "drop"
    )
}

# ---- Normalization helpers for plotting (A/(A+B)) ----
ratio_ab <- function(a, b) {
  if_else((a + b) > 0, a / (a + b), NA_real_)
}

summarize_cis_trans_ab <- function(dat,
                                  count_col = "count",
                                  group_cols = c("batch", "time_point", "DSB"),
                                  cis_trans_col = "cis_trans",
                                  combo_col = "combo") {
  if (!all(group_cols %in% names(dat))) {
    missing <- setdiff(group_cols, names(dat))
    stop("Missing grouping columns in data: ", paste(missing, collapse = ", "))
  }
  if (!(cis_trans_col %in% names(dat))) stop("Missing cis/trans column: ", cis_trans_col)
  if (!(combo_col %in% names(dat))) stop("Missing combo column: ", combo_col)
  if (!(count_col %in% names(dat))) stop("Missing count column: ", count_col)

  dat %>%
    filter(.data[[combo_col]] %in% combos_4) %>%
    group_by(across(all_of(group_cols))) %>%
    summarise(
      Cis_Counts = sum(.data[[count_col]][.data[[cis_trans_col]] == "CIS"], na.rm = TRUE),
      Trans_Counts = sum(.data[[count_col]][.data[[cis_trans_col]] == "TRANS"], na.rm = TRUE),
      CisTrans_Total = Cis_Counts + Trans_Counts,
      Fraction_Cis = ratio_ab(Cis_Counts, Trans_Counts),
      Fraction_Trans = ratio_ab(Trans_Counts, Cis_Counts),
      Percent_Cis = 100 * Fraction_Cis,
      Percent_Trans = 100 * Fraction_Trans,
      .groups = "drop"
    )
}

summarize_intact_ssa_ab <- function(dat,
                                   count_col = "count",
                                   group_cols = c("batch", "time_point", "DSB"),
                                   repeat_col = "repeat",
                                   combo_col = "combo") {
  if (!all(group_cols %in% names(dat))) {
    missing <- setdiff(group_cols, names(dat))
    stop("Missing grouping columns in data: ", paste(missing, collapse = ", "))
  }
  if (!(repeat_col %in% names(dat))) stop("Missing repeat column: ", repeat_col)
  if (!(combo_col %in% names(dat))) stop("Missing combo column: ", combo_col)
  if (!(count_col %in% names(dat))) stop("Missing count column: ", count_col)

  dat %>%
    filter(.data[[combo_col]] %in% combos_4) %>%
    group_by(across(all_of(group_cols))) %>%
    summarise(
      Intact_Counts = sum(.data[[count_col]][stringr::str_to_upper(as.character(.data[[repeat_col]])) == "INTACT"], na.rm = TRUE),
      SSA_Counts = sum(.data[[count_col]][stringr::str_to_upper(as.character(.data[[repeat_col]])) == "SSA"], na.rm = TRUE),
      IntactSSA_Total = Intact_Counts + SSA_Counts,
      Fraction_Intact = ratio_ab(Intact_Counts, SSA_Counts),
      Fraction_SSA = ratio_ab(SSA_Counts, Intact_Counts),
      Percent_Intact = 100 * Fraction_Intact,
      Percent_SSA = 100 * Fraction_SSA,
      .groups = "drop"
    )
}

summarize_allele_frequency <- function(dat,
                                      count_col = "count",
                                      group_cols = c("batch", "time_point", "DSB"),
                                      allele_col = "allele",
                                      combo_col = "combo") {
  if (!all(group_cols %in% names(dat))) {
    missing <- setdiff(group_cols, names(dat))
    stop("Missing grouping columns in data: ", paste(missing, collapse = ", "))
  }
  if (!(allele_col %in% names(dat))) stop("Missing allele column: ", allele_col)
  if (!(combo_col %in% names(dat))) stop("Missing combo column: ", combo_col)
  if (!(count_col %in% names(dat))) stop("Missing count column: ", count_col)

  dat %>%
    filter(.data[[combo_col]] %in% combos_4) %>%
    group_by(across(all_of(group_cols)), .data[[allele_col]]) %>%
    summarise(Allele_Counts = sum(.data[[count_col]], na.rm = TRUE), .groups = "drop") %>%
    group_by(across(all_of(group_cols))) %>%
    mutate(
      Total_Counts = sum(Allele_Counts, na.rm = TRUE),
      Allele_Frequency = if_else(Total_Counts > 0, Allele_Counts / Total_Counts, NA_real_)
    ) %>%
    ungroup()
}

## 1. Read and combine ALL CSVs

Reads all `_summary.csv` files from `pd_folder` and combines them into a single master table `dat_raw`.

Outputs created:
- `dat_raw`: combined rows from all CSVs
- `dat_pd`: biological PD alleles (control alleles removed)
- `spike_factors`, `nocut_denoms`: normalization diagnostics

In [ ]:
pd_folder <- "C:/Users/dunnmk/University of Michigan Dropbox/MED-WILSONTELAB/wilsontelab box/Common/Projects/Yeast Aim 3/Sequencing/PD Data"

# IMPORTANT:
# This report is PD-only. Do not point it at 3C summary folders.
folders <- c(pd_folder)

files <- unlist(lapply(
  folders,
  function(folder) list.files(
    folder,
    pattern = "(_summary|_collapsed_summary)\\.csv$",
    full.names = TRUE
  )
))

raw_list <- lapply(files, function(f) {
  dat <- readr::read_csv(f, show_col_types = FALSE)
  if (!("repeat" %in% names(dat))) {
    dat <- dat %>% mutate(`repeat` = "ALL")
  }
  if (!("replicate" %in% names(dat))) {
    dat <- dat %>% mutate(replicate = "ALL")
  }
  if (!("alignment_name" %in% names(dat))) {
    dat <- dat %>% mutate(alignment_name = NA_character_)
  }
  dat %>%
    mutate(
      source_path = f,
      source_dir = dirname(f),
      source_file = basename(f),
      `repeat` = as.character(`repeat`),
      replicate = as.character(replicate)
    )
})

dat_raw <- bind_rows(raw_list)
dat_raw <- dat_raw %>%
  mutate(batch = factor(batch, levels = sort(unique(batch))))

# Make sure key columns are character (helps joins and avoids factor surprises)
dat_raw <- dat_raw %>%
  mutate(
    allele = as.character(allele),
    combo = as.character(combo),
    cis_trans = as.character(cis_trans),
    `repeat` = as.character(`repeat`)
  )

# ---- PD-only normalization controls ----
# 1) Between-batch normalization using the 1% spike-in allele.
# 2) Within-repeat normalization using the NO_CUT control allele.
spike_totals <- dat_raw %>%
  filter(allele == SPIKEIN_ALLELE, combo %in% combos_4) %>%
  group_by(batch, time_point) %>%
  summarise(SpikeIn_Counts = sum(count, na.rm = TRUE), .groups = "drop")

spike_ref <- spike_totals %>%
  filter(SpikeIn_Counts > 0) %>%
  summarise(ref = median(SpikeIn_Counts, na.rm = TRUE)) %>%
  pull(ref)

if (length(spike_ref) == 0 || is.na(spike_ref)) {
  spike_ref <- NA_real_
}

spike_factors <- spike_totals %>%
  mutate(
    SpikeIn_Scale = if_else(!is.na(spike_ref) & SpikeIn_Counts > 0, spike_ref / SpikeIn_Counts, 1.0),
    SpikeIn_Flag = case_when(
      is.na(spike_ref) ~ "no_spike_reference",
      SpikeIn_Counts <= 0 ~ "missing_or_zero_spike",
      TRUE ~ "ok"
    )
  )

dat_raw <- dat_raw %>%
  left_join(spike_factors %>% select(batch, time_point, SpikeIn_Scale, SpikeIn_Flag),
            by = c("batch", "time_point")) %>%
  mutate(
    SpikeIn_Scale = tidyr::replace_na(SpikeIn_Scale, 1.0),
    SpikeIn_Flag = tidyr::replace_na(SpikeIn_Flag, "missing_or_zero_spike"),
    count_spike_scaled = count * SpikeIn_Scale
  )

nocut_denoms <- dat_raw %>%
  filter(allele == NOCUT_ALLELE, combo %in% combos_4) %>%
  group_by(batch, time_point, DSB, combo, `repeat`) %>%
  summarise(NoCut_Denom = sum(count_spike_scaled, na.rm = TRUE), .groups = "drop")

dat_raw <- dat_raw %>%
  left_join(nocut_denoms, by = c("batch", "time_point", "DSB", "combo", "repeat")) %>%
  mutate(
    # Doubly-normalized counts: spike-in scaled, then normalized to the matched NO_CUT denominator.
    count_norm = if_else(!is.na(NoCut_Denom) & NoCut_Denom > 0,
                         count_spike_scaled / NoCut_Denom,
                         NA_real_)
  )

# PD biological rows only (exclude the spike-in and NO_CUT control alleles)
dat_pd <- dat_raw %>%
  filter(!(allele %in% CONTROL_ALLELES))

str(dat_raw)

# Quick diagnostic: what time_points are coming from which directory?
dat_raw %>%
  count(source_dir, time_point, sort = TRUE, name = "n_rows") %>%
  head(30) %>%
  knitr::kable(caption = "Diagnostic: rows by source_dir × time_point (top 30)")

dat_raw %>% head(10) %>%
  knitr::kable(caption = "Preview: combined raw `_summary.csv` rows (first 10)")

# Quick QC: spike-in totals and applied scale factors (PD only)
spike_factors %>%
  arrange(time_point, batch) %>%
  knitr::kable(caption = "Spike-in totals and scaling factors by batch × time_point")

nocut_denoms %>%
  arrange(time_point, batch, DSB, `repeat`, combo) %>%
  head(20) %>%
  knitr::kable(caption = "NO_CUT denominators (spike-scaled) by batch × time_point × DSB × repeat × combo (first 20)")

## 2. Aggregate counts and percentages

Purpose: calculate percentages from aggregated counts.

In [ ]:
dat_agg_counts <- summarize_combo4_counts_pct(
  dat_pd,
  count_col = "count_spike_scaled",
  group_cols = c("batch", "time_point", "DSB"),
  combo_col = "combo"
)

str(dat_agg_counts)

dat_agg_counts %>%
  head(10) %>%
  knitr::kable(caption = "Aggregated counts + derived percentages by batch × time_point × DSB (first 10)")

## 2.5 Batch-level QC plots (counts + composition)

Purpose: quick sanity-check summaries *before* allele-level cis/trans normalization.

In [ ]:
# Aggregate from the master raw table
agg_qc <- dat_pd %>%
  group_by(batch, time_point, DSB) %>%
  summarize_combo4_counts_pct(
    count_col = "count_spike_scaled",
    group_cols = c("batch", "time_point", "DSB"),
    combo_col = "combo"
  )

# From here on in QC, compute CIS/TRANS using ONLY the 4 combos of interest:
#   CIS   := A_to_B + C_to_D
#   TRANS := A_to_D + C_to_B
# and define percents relative to the total of these 4 combos.
#
# IMPORTANT: we intentionally DO NOT group by `DSB` here.
# In this dataset, `DSB` can include levels like "TRANS", and grouping by it
# will split CIS and TRANS into different facets and produce misleading 100% bars.
agg_qc_combo4 <- dat_raw %>%
  group_by(batch, time_point) %>%
  summarise(
    Total_All = sum(count_spike_scaled),
    A_to_B = sum(count_spike_scaled[combo == "A_to_B"]),
    C_to_D = sum(count_spike_scaled[combo == "C_to_D"]),
    A_to_D = sum(count_spike_scaled[combo == "A_to_D"]),
    C_to_B = sum(count_spike_scaled[combo == "C_to_B"]),
    .groups = "drop"
  ) %>%
  mutate(
    Cis_Counts = A_to_B + C_to_D,
    Trans_Counts = A_to_D + C_to_B,
    Total_4Combos = Cis_Counts + Trans_Counts,
    Excluded_Counts = pmax(Total_All - Total_4Combos, 0),
    Percent_Cis = if_else(Total_4Combos > 0, 100 * Cis_Counts / Total_4Combos, NA_real_),
    Percent_Trans = if_else(Total_4Combos > 0, 100 * Trans_Counts / Total_4Combos, NA_real_),
    Percent_A_to_B_in_Cis = if_else(Cis_Counts > 0, 100 * A_to_B / Cis_Counts, NA_real_),
    Percent_C_to_D_in_Cis = if_else(Cis_Counts > 0, 100 * C_to_D / Cis_Counts, NA_real_),
    Percent_A_to_D_in_Trans = if_else(Trans_Counts > 0, 100 * A_to_D / Trans_Counts, NA_real_),
    Percent_C_to_B_in_Trans = if_else(Trans_Counts > 0, 100 * C_to_B / Trans_Counts, NA_real_)
  )

# Quick diagnostic table: if Excluded_Counts is large, then the 4-combo definition is omitting real counts.
agg_qc_combo4 %>%
  arrange(time_point, batch) %>%
  transmute(
    batch, time_point,
    Total_All,
    Total_4Combos,
    Excluded_Counts,
    Percent_Cis,
    Percent_Trans
  ) %>%
  head(20) %>%
  knitr::kable(caption = "QC (4-combo definition): totals vs excluded counts, plus CIS%/TRANS% (first 20 rows)")

# Formatting helpers (avoid extra package dependencies)
comma_label <- function(x) {
  ifelse(is.na(x), NA_character_, formatC(x, format = "f", digits = 0, big.mark = ","))
}

pct_label <- function(x, digits = 1) {
  ifelse(is.na(x), NA_character_, paste0(round(x, digits), "%"))
}

## 2.6 A/(A+B) normalization plots (CIS/TRANS and INTACT/SSA)

These summaries use the $A/(A+B)$ form to match expected trends:

- $\%\text{CIS} = \frac{\text{CIS}}{\text{CIS}+\text{TRANS}} \times 100$
- $\%\text{TRANS} = \frac{\text{TRANS}}{\text{CIS}+\text{TRANS}} \times 100$
- $\%\text{INTACT} = \frac{\text{INTACT}}{\text{INTACT}+\text{SSA}} \times 100$
- $\%\text{SSA} = \frac{\text{SSA}}{\text{INTACT}+\text{SSA}} \times 100$

In [ ]:
group_cols_ab <- c("batch", "time_point", "DSB")

cis_trans_ab <- summarize_cis_trans_ab(
  dat_pd,
  count_col = "count_norm",
  group_cols = group_cols_ab,
  cis_trans_col = "cis_trans",
  combo_col = "combo"
)

intact_ssa_ab <- summarize_intact_ssa_ab(
  dat_pd,
  count_col = "count_norm",
  group_cols = group_cols_ab,
  repeat_col = "repeat",
  combo_col = "combo"
)

ab_joined <- cis_trans_ab %>%
  left_join(intact_ssa_ab, by = group_cols_ab)

ab_joined %>%
  arrange(batch, DSB, time_point) %>%
  head(20) %>%
  knitr::kable(caption = "Preview: A/(A+B) normalized summaries (first 20 rows)")

# Plot 1: CIS vs TRANS over time (each facet is batch × DSB)
cis_trans_long <- ab_joined %>%
  select(any_of(group_cols_ab), Percent_Cis, Percent_Trans) %>%
  pivot_longer(
    cols = c(Percent_Cis, Percent_Trans),
    names_to = "Class",
    values_to = "Percent"
  ) %>%
  mutate(
    Class = recode(Class, Percent_Cis = "CIS", Percent_Trans = "TRANS"),
    time_point = as.numeric(as.character(time_point))
  )

if (nrow(cis_trans_long) > 0 && any(!is.na(cis_trans_long$Percent))) {
  p_cis_trans_time <- ggplot(cis_trans_long, aes(x = time_point, y = Percent, color = Class, group = Class)) +
    geom_line(linewidth = 1.1) +
    geom_point(size = 2.5) +
    facet_grid(DSB ~ batch, scales = "free_x") +
    scale_y_continuous(limits = c(0, 100)) +
    theme_bw() +
    labs(
      title = "CIS vs TRANS over time (A/(A+B) normalization)",
      x = "Time point",
      y = "% of (CIS + TRANS)",
      color = ""
    )
  print(p_cis_trans_time)
}

# Plot 2: INTACT vs SSA over time (each facet is batch × DSB)
intact_ssa_long <- ab_joined %>%
  select(any_of(group_cols_ab), Percent_Intact, Percent_SSA) %>%
  pivot_longer(
    cols = c(Percent_Intact, Percent_SSA),
    names_to = "Class",
    values_to = "Percent"
  ) %>%
  mutate(
    Class = recode(Class, Percent_Intact = "INTACT", Percent_SSA = "SSA"),
    time_point = as.numeric(as.character(time_point))
  )

if (nrow(intact_ssa_long) > 0 && any(!is.na(intact_ssa_long$Percent))) {
  p_intact_ssa_time <- ggplot(intact_ssa_long, aes(x = time_point, y = Percent, color = Class, group = Class)) +
    geom_line(linewidth = 1.1) +
    geom_point(size = 2.5) +
    facet_grid(DSB ~ batch, scales = "free_x") +
    scale_y_continuous(limits = c(0, 100)) +
    theme_bw() +
    labs(
      title = "INTACT vs SSA over time (A/(A+B) normalization)",
      x = "Time point",
      y = "% of (INTACT + SSA)",
      color = ""
    )
  print(p_intact_ssa_time)
}

## 2.7 Allele frequency plots (normalized counts)

This section plots allele frequencies computed from the doubly-normalized counts (`count_norm`).

In [ ]:
# Compute allele frequencies per batch (and time_point × DSB) using normalized counts.
dat_allele_freq <- summarize_allele_frequency(
  dat_pd,
  count_col = "count_norm",
  group_cols = c("batch", "time_point", "DSB"),
  allele_col = "allele",
  combo_col = "combo"
)

freq_label <- function(x, accuracy = 0.1) {
  ifelse(is.na(x), NA_character_, scales::percent(x, accuracy = accuracy))
}

# Only compare these two timepoints on the same plot (per your request)
timepoints_compare <- c(0, 180)

# Per-batch plots (each batch printed as a separate plot)
plot_allele_frequency_by_batch <- function(df) {
  batches <- df %>% distinct(batch) %>% arrange(batch) %>% pull(batch)
  for (b in batches) {
    df_batch <- df %>%
      filter(batch == b) %>%
      mutate(
        time_point = as.numeric(as.character(time_point)),
        allele = as.character(allele)
      )
    if (nrow(df_batch) == 0 || all(is.na(df_batch$Allele_Frequency))) next

    dsbs <- df_batch %>% distinct(DSB) %>% arrange(DSB) %>% pull(DSB)

    # ---- Bar plots (one plot per DSB): time_point 0 and 180 on the same plot ----
    for (d in dsbs) {
      df_plot <- df_batch %>%
        filter(DSB == d, time_point %in% timepoints_compare)
      if (nrow(df_plot) == 0 || all(is.na(df_plot$Allele_Frequency))) next

      # If one of the requested timepoints is missing, skip to avoid misleading comparisons.
      have_tps <- sort(unique(df_plot$time_point))
      if (!all(timepoints_compare %in% have_tps)) next

      # Order alleles (globally within this batch+DSB) by max frequency for readability
      df_plot <- df_plot %>%
        group_by(allele) %>%
        mutate(.max_freq = max(Allele_Frequency, na.rm = TRUE)) %>%
        ungroup() %>%
        mutate(allele = forcats::fct_reorder(allele, .max_freq, .desc = TRUE))

      df_plot <- df_plot %>%
        mutate(time_point = factor(time_point, levels = timepoints_compare))

      dodge <- position_dodge(width = 0.9)
      p_bar <- ggplot(df_plot, aes(x = allele, y = Allele_Frequency, fill = time_point)) +
        geom_col(width = 0.85, position = dodge) +
        geom_text(
          aes(label = freq_label(Allele_Frequency, accuracy = 0.1)),
          position = dodge,
          vjust = -0.25,
          size = 3,
          na.rm = TRUE
        ) +
        scale_y_continuous(
          labels = scales::percent_format(accuracy = 1),
          limits = c(0, 1),
          expand = expansion(mult = c(0, 0.18))
        ) +
        theme_bw() +
        theme(axis.text.x = element_text(angle = 90, hjust = 0.5, vjust = 0.5, size = 9)) +
        labs(
          title = paste0("Allele frequency by allele | Batch: ", b, " | DSB: ", d, " (normalized counts)"),
          x = "Allele",
          y = "Allele frequency",
          fill = "Time point"
        )
      print(p_bar)
    }
  }
}

plot_allele_frequency_by_batch(dat_allele_freq)

# Overall allele frequency pooled across batches (weighted by normalized counts)
dat_allele_freq_overall <- dat_pd %>%
  filter(combo %in% combos_4) %>%
  group_by(time_point, DSB, allele) %>%
  summarise(Allele_Counts = sum(count_norm, na.rm = TRUE), .groups = "drop") %>%
  group_by(time_point, DSB) %>%
  mutate(
    Total_Counts = sum(Allele_Counts, na.rm = TRUE),
    Allele_Frequency = if_else(Total_Counts > 0, Allele_Counts / Total_Counts, NA_real_)
  ) %>%
  ungroup()

if (nrow(dat_allele_freq_overall) > 0 && any(!is.na(dat_allele_freq_overall$Allele_Frequency))) {
  df_overall <- dat_allele_freq_overall %>%
    mutate(
      time_point = as.numeric(as.character(time_point)),
      allele = as.character(allele)
    )

  dsbs_overall <- df_overall %>% distinct(DSB) %>% arrange(DSB) %>% pull(DSB)

  # Bar plots: one plot per DSB, time_point 0 and 180 on the same plot
  for (d in dsbs_overall) {
    df_plot <- df_overall %>%
      filter(DSB == d, time_point %in% timepoints_compare)
    if (nrow(df_plot) == 0 || all(is.na(df_plot$Allele_Frequency))) next

    have_tps <- sort(unique(df_plot$time_point))
    if (!all(timepoints_compare %in% have_tps)) next

    df_plot <- df_plot %>%
      group_by(allele) %>%
      mutate(.max_freq = max(Allele_Frequency, na.rm = TRUE)) %>%
      ungroup() %>%
      mutate(allele = forcats::fct_reorder(allele, .max_freq, .desc = TRUE))

    df_plot <- df_plot %>%
      mutate(time_point = factor(time_point, levels = timepoints_compare))

    dodge <- position_dodge(width = 0.9)
    p_overall_bar <- ggplot(df_plot, aes(x = allele, y = Allele_Frequency, fill = time_point)) +
      geom_col(width = 0.85, position = dodge) +
      geom_text(
        aes(label = freq_label(Allele_Frequency, accuracy = 0.1)),
        position = dodge,
        vjust = -0.25,
        size = 3,
        na.rm = TRUE
      ) +
      scale_y_continuous(
        labels = scales::percent_format(accuracy = 1),
        limits = c(0, 1),
        expand = expansion(mult = c(0, 0.18))
      ) +
      theme_bw() +
      theme(axis.text.x = element_text(angle = 90, hjust = 0.5, vjust = 0.5, size = 9)) +
      labs(
        title = paste0("Overall allele frequency pooled across batches | DSB: ", d, " (normalized counts)"),
        x = "Allele",
        y = "Allele frequency",
        fill = "Time point"
      )
    print(p_overall_bar)
  }
}

## 2.8 Pearson correlation: allele frequency changes by combo (t=0 vs t=180)

This section compares allele frequencies between time point 0 and 180, **separately for each combo**, using Pearson correlation.

In [ ]:
# Allele frequencies stratified by combo as well
dat_allele_freq_combo <- summarize_allele_frequency(
  dat_pd,
  count_col = "count_norm",
  group_cols = c("batch", "time_point", "DSB", "combo"),
  allele_col = "allele",
  combo_col = "combo"
) %>%
  mutate(time_point = as.numeric(as.character(time_point))) %>%
  filter(time_point %in% timepoints_compare)

# Wide format for correlation/scatter
dat_wide <- dat_allele_freq_combo %>%
  select(batch, DSB, combo, allele, time_point, Allele_Frequency) %>%
  mutate(tp = paste0("t", time_point)) %>%
  select(-time_point) %>%
  tidyr::pivot_wider(names_from = tp, values_from = Allele_Frequency)

# Correlation stats per facet
cor_stats <- dat_wide %>%
  group_by(batch, DSB, combo) %>%
  summarise(
    n_pairs = sum(!is.na(t0) & !is.na(t180)),
    r = if_else(n_pairs >= 3, cor(t0, t180, use = "complete.obs", method = "pearson"), NA_real_),
    .groups = "drop"
  ) %>%
  mutate(
    label = if_else(
      is.na(r),
      paste0("n=", n_pairs),
      paste0("r=", round(r, 3), "\n", "n=", n_pairs)
    ),
    x = 0.02,
    y = 0.98
  )

plot_cor_by_batch <- function(df_wide, df_stats) {
  batches <- df_wide %>% distinct(batch) %>% arrange(batch) %>% pull(batch)
  for (b in batches) {
    dfb <- df_wide %>% filter(batch == b)
    if (nrow(dfb) == 0) next

    statsb <- df_stats %>% filter(batch == b)

    p <- ggplot(dfb, aes(x = t0, y = t180)) +
      geom_abline(slope = 1, intercept = 0, linewidth = 0.7, linetype = "dashed", color = "grey50") +
      geom_point(alpha = 0.75, size = 1.8) +
      facet_grid(combo ~ DSB) +
      geom_text(
        data = statsb,
        aes(x = x, y = y, label = label),
        inherit.aes = FALSE,
        hjust = 0,
        vjust = 1,
        size = 3
      ) +
      scale_x_continuous(labels = scales::percent_format(accuracy = 1), limits = c(0, 1)) +
      scale_y_continuous(labels = scales::percent_format(accuracy = 1), limits = c(0, 1)) +
      theme_bw() +
      labs(
        title = paste0("Pearson correlation of allele frequencies (t=0 vs t=180) | Batch: ", b),
        x = "Allele frequency at t=0",
        y = "Allele frequency at t=180"
      )
    print(p)
  }
}

plot_cor_by_batch(dat_wide, cor_stats)

# Also print a pooled-across-batches view
dat_wide_overall <- dat_wide %>%
  group_by(DSB, combo, allele) %>%
  summarise(
    t0 = sum(t0, na.rm = TRUE),
    t180 = sum(t180, na.rm = TRUE),
    .groups = "drop"
  )

cor_stats_overall <- dat_wide_overall %>%
  group_by(DSB, combo) %>%
  summarise(
    n_pairs = sum(!is.na(t0) & !is.na(t180)),
    r = if_else(n_pairs >= 3, cor(t0, t180, use = "complete.obs", method = "pearson"), NA_real_),
    .groups = "drop"
  ) %>%
  mutate(
    label = if_else(
      is.na(r),
      paste0("n=", n_pairs),
      paste0("r=", round(r, 3), "\n", "n=", n_pairs)
    ),
    x = 0.02,
    y = 0.98
  )

if (nrow(dat_wide_overall) > 0) {
  p_overall <- ggplot(dat_wide_overall, aes(x = t0, y = t180)) +
    geom_abline(slope = 1, intercept = 0, linewidth = 0.7, linetype = "dashed", color = "grey50") +
    geom_point(alpha = 0.75, size = 1.8) +
    facet_grid(combo ~ DSB) +
    geom_text(
      data = cor_stats_overall,
      aes(x = x, y = y, label = label),
      inherit.aes = FALSE,
      hjust = 0,
      vjust = 1,
      size = 3
    ) +
    scale_x_continuous(labels = scales::percent_format(accuracy = 1), limits = c(0, 1)) +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1), limits = c(0, 1)) +
    theme_bw() +
    labs(
      title = "Pearson correlation of allele frequencies (t=0 vs t=180) pooled across batches",
      x = "Allele frequency at t=0",
      y = "Allele frequency at t=180"
    )
  print(p_overall)
}